In [ ]:
import subprocess, sys, os, warnings, time

try:
    import optuna, xgboost, pywt
except ImportError:
    print("Missing dependencies. Installing: optuna, xgboost, PyWavelets")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "xgboost", "PyWavelets"])
    import optuna, xgboost, pywt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.signal import savgol_filter
from scipy.stats import linregress
import requests, io
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Maintain Strict Reproducibility
np.random.seed(42)
m_list = ['PLS', 'Ridge', 'ElasticNet', 'SVR', 'XGBoost']
OPTUNA_BUDGET = {'PLS': 30, 'Ridge': 30, 'ElasticNet': 30, 'SVR': 50, 'XGBoost': 80}

# --- DATA LOADING ---
def get_data():
    lf = [f for f in os.listdir('.') if f.endswith('.csv') and 'gasoline' in f.lower()]
    if lf:
        df = pd.read_csv(lf[0])
    else:
        r = requests.get("https://raw.githubusercontent.com/elnegmelnegm/gasoline/main/gasoline.csv", timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))

    df.dropna(axis=1, how='all', inplace=True)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    tc = next((c for c in df.columns if 'octane' in c.lower()), df.columns[-1])
    df.dropna(subset=[tc], inplace=True)
    Y = df[[tc]]
    X = df.drop(columns=[tc])
    try:
        w = np.array([float(c) for c in X.columns])
    except:
        w = np.linspace(900, 1700, X.shape[1])
    return X, Y, w

X_raw, Y_raw, wavs = get_data()

# --- WINDOW SCENARIOS ---
window_scenarios = {
    "Full Spectrum": (wavs.min(), wavs.max()),
    "Broad Structure": (1100, 1350),
    "Core CH": (1150, 1250),
    "Narrow CH": (1180, 1220)
}

# --- PREPROCESSING MODULE ---
class SpPrep(BaseEstimator, TransformerMixin):
    def __init__(self, method='none', window=15, w_family='db4'):
        self.method = method; self.window = window; self.w_family = w_family
        self.ref_mean = None

    def fit(self, X, y=None):
        if self.method == 'msc': self.ref_mean = np.mean(X, axis=0)
        return self

    def _get_odd_window(self, max_len, polyorder):
        w = min(self.window, max_len)
        if w % 2 == 0: w -= 1
        w = max(w, polyorder + 2)
        if w % 2 == 0: w += 1
        if w > max_len: w = max_len if max_len % 2 == 1 else max_len - 1
        return w

    def transform(self, X):
        X_n = np.array(X).copy()
        if self.method == 'snv':
            stds = np.std(X_n, axis=1, keepdims=True)
            stds[stds == 0] = 1e-8
            X_n = (X_n - np.mean(X_n, axis=1, keepdims=True)) / stds
        elif self.method == 'msc':
            for i in range(X_n.shape[0]):
                slope, intercept, _, _, _ = linregress(self.ref_mean, X_n[i])
                if slope == 0: slope = 1e-8
                X_n[i] = (X_n[i] - intercept) / slope
        elif self.method == 'deriv1':
            X_n = savgol_filter(X_n, self._get_odd_window(X_n.shape[1], 2), 2, deriv=1, axis=1)
        elif self.method == 'deriv2':
            X_n = savgol_filter(X_n, self._get_odd_window(X_n.shape[1], 3), 3, deriv=2, axis=1)
        elif self.method == 'wavelet':
            denoised = np.zeros_like(X_n)
            for i in range(X_n.shape[0]):
                coeffs = pywt.wavedec(X_n[i], self.w_family, level=2)
                sigma = np.median(np.abs(coeffs[-1])) / 0.6745
                uthresh = sigma * np.sqrt(2 * np.log(len(X_n[i])))
                coeffs_t = [coeffs[0]] + [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs[1:]]
                denoised[i] = pywt.waverec(coeffs_t, self.w_family)[:X_n.shape[1]]
            X_n = denoised
        return pd.DataFrame(X_n, index=X.index, columns=X.columns) if isinstance(X, pd.DataFrame) else X_n

# --- SINGLE SOURCE OF TRUTH: OBJECTIVE FUNCTION ---
def make_objective(n, X_data, Y_data, cv):
    def obj(t):
        p_method = t.suggest_categorical('p_method', ['none', 'snv', 'msc', 'deriv1', 'deriv2', 'wavelet'])
        p_window = 15; p_wavelet = 'db4'
        use_scaler = t.suggest_categorical('use_scaler', [True, False])
        if p_method in ['deriv1', 'deriv2']: p_window = t.suggest_int('p_window', 5, 25, step=2)
        elif p_method == 'wavelet': p_wavelet = t.suggest_categorical('p_wavelet', ['db2', 'db4', 'sym4'])

        if n=='PLS':
            nl = min(int(len(X_data) * 2 / 3) - 1, X_data.shape[1])
            e = PLSRegression(n_components=t.suggest_int('n', 1, max(1, min(15, nl))), scale=False)
        elif n=='Ridge': e = Ridge(alpha=t.suggest_float('a', 0.01, 100, log=True), random_state=42)
        elif n=='ElasticNet': e = ElasticNet(alpha=t.suggest_float('a', 0.01, 10, log=True), l1_ratio=t.suggest_float('l1', 1e-4, 1.0), random_state=42)
        elif n=='SVR':
            kernel = t.suggest_categorical('kernel', ['linear', 'poly', 'rbf'])
            params = {'C': t.suggest_float('c', 0.1, 100, log=True), 'epsilon': t.suggest_float('eps', 0.01, 0.5), 'kernel': kernel}
            if kernel == 'poly': params['degree'] = t.suggest_int('degree', 2, 4)
            e = SVR(**params)
        elif n=='XGBoost':
            e = XGBRegressor(n_estimators=t.suggest_int('n', 50, 300), max_depth=t.suggest_int('d', 2, 8), learning_rate=t.suggest_float('lr', 0.01, 0.3, log=True), subsample=t.suggest_float('subsample', 0.6, 1.0), colsample_bytree=t.suggest_float('colsample', 0.5, 1.0), reg_alpha=t.suggest_float('reg_alpha', 1e-3, 10, log=True), reg_lambda=t.suggest_float('reg_lambda', 1e-3, 10, log=True), min_child_weight=t.suggest_int('mcw', 1, 10), n_jobs=1, verbosity=0, random_state=42)

        steps = [('i', SimpleImputer(strategy='mean')), ('pr', SpPrep(method=p_method, window=p_window, w_family=p_wavelet))]
        if use_scaler: steps.append(('s', StandardScaler()))
        steps.append(('e', e))
        return cross_val_score(Pipeline(steps), X_data.values, Y_data.values.ravel(), cv=cv, scoring='r2', n_jobs=1).mean()
    return obj

# --- PHASE 1: NESTED CV PIPELINE ---
def run_nested_pipeline(X, Y):
    st = {m: {'oof_preds': np.zeros(len(Y)), 'train_time': 0, 'best_preps': [], 'fold_r2s': [], 'fold_rmses': [], 'fold_maes': [], 'l1_ratios': []} for m in m_list}
    outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

    for n in m_list:
        print(f"  Training {n}...")
        t0 = time.time()
        for fold_i, (train_idx, test_idx) in enumerate(outer_cv.split(X)):
            X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
            Y_tr, Y_te = Y.iloc[train_idx], Y.iloc[test_idx]

            obj = make_objective(n, X_tr, Y_tr, inner_cv)
            study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(obj, n_trials=OPTUNA_BUDGET[n], n_jobs=1)
            bp = study.best_params

            p_m = bp.get('p_method', 'none'); p_w = bp.get('p_window', 15); p_wf = bp.get('p_wavelet', 'db4'); use_s = bp.get('use_scaler', True)
            st[n]['best_preps'].append(f"{p_m} ({'+S' if use_s else '-S'})")
            if n == 'ElasticNet': st[n]['l1_ratios'].append(bp['l1'])

            if n=='PLS': e = PLSRegression(n_components=bp['n'], scale=False)
            elif n=='Ridge': e = Ridge(alpha=bp['a'], random_state=42)
            elif n=='ElasticNet': e = ElasticNet(alpha=bp['a'], l1_ratio=bp['l1'], random_state=42)
            elif n=='SVR': e = SVR(C=bp['c'], epsilon=bp['eps'], kernel=bp['kernel'], degree=bp.get('degree', 3))
            elif n=='XGBoost': e = XGBRegressor(n_estimators=bp['n'], max_depth=bp['d'], learning_rate=bp['lr'], subsample=bp['subsample'], colsample_bytree=bp['colsample'], reg_alpha=bp['reg_alpha'], reg_lambda=bp['reg_lambda'], min_child_weight=bp['mcw'], n_jobs=1, verbosity=0, random_state=42)

            final_steps = [('i', SimpleImputer(strategy='mean')), ('pr', SpPrep(method=p_m, window=p_w, w_family=p_wf))]
            if use_s: final_steps.append(('s', StandardScaler()))
            final_steps.append(('e', e))
            final_pipe = Pipeline(final_steps)

            final_pipe.fit(X_tr.values, Y_tr.values.ravel())
            fold_preds = final_pipe.predict(X_te.values)
            st[n]['oof_preds'][test_idx] = fold_preds

            st[n]['fold_r2s'].append(r2_score(Y_te.values.ravel(), fold_preds))
            st[n]['fold_rmses'].append(np.sqrt(mean_squared_error(Y_te.values.ravel(), fold_preds)))
            st[n]['fold_maes'].append(mean_absolute_error(Y_te.values.ravel(), fold_preds))

        st[n]['train_time'] = time.time() - t0
        # Restore aggregate metrics explicitly for evaluation & verification
        st[n]['oof_r2'] = r2_score(Y.values.ravel(), st[n]['oof_preds'])
        st[n]['oof_rmse'] = np.sqrt(mean_squared_error(Y.values.ravel(), st[n]['oof_preds']))
        st[n]['oof_mae'] = mean_absolute_error(Y.values.ravel(), st[n]['oof_preds'])

    return st

# --- PHASE 2: FINAL PRODUCTION TUNING ---
def run_production_tuning(X, Y):
    prod_params = {}
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    for n in m_list:
        print(f"    Production tuning {n}...")
        obj = make_objective(n, X, Y, inner_cv)
        study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(obj, n_trials=OPTUNA_BUDGET[n], n_jobs=1)
        prod_params[n] = study.best_params
    return prod_params

# Execute Core Pipeline
results = {}
production_params = {}
for scen_name, (lo, hi) in window_scenarios.items():
    print(f"\nEvaluating Scenario: {scen_name} ({lo}-{hi} nm)")
    mask = (wavs >= lo) & (wavs <= hi)
    results[scen_name] = run_nested_pipeline(X_raw.loc[:, mask], Y_raw)

    print(f"  Running Final Production Tuning for {scen_name}...")
    production_params[scen_name] = run_production_tuning(X_raw.loc[:, mask], Y_raw)

# --- ROBUST STATISTICAL TESTS ---
def bootstrap_r2_diff(y_true, preds_a, preds_b, n_boot=5000, seed=42):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(y_true))
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        b = rng.choice(idx, size=len(idx), replace=True)
        r2_a = r2_score(y_true[b], preds_a[b])
        r2_b = r2_score(y_true[b], preds_b[b])
        diffs[i] = r2_a - r2_b
    ci_lower, ci_upper = np.percentile(diffs, [2.5, 97.5])
    return np.mean(diffs), ci_lower, ci_upper, (ci_lower > 0) or (ci_upper < 0)

def permutation_test_r2(y_true, preds_a, preds_b, n_perm=5000, seed=42):
    rng = np.random.RandomState(seed)
    obs_diff = r2_score(y_true, preds_a) - r2_score(y_true, preds_b)
    count = 0
    for _ in range(n_perm):
        swap = rng.binomial(1, 0.5, size=len(y_true)).astype(bool)
        perm_a = np.where(swap, preds_b, preds_a)
        perm_b = np.where(swap, preds_a, preds_b)
        perm_diff = r2_score(y_true, perm_a) - r2_score(y_true, perm_b)
        if perm_diff >= obs_diff:
            count += 1
    return obs_diff, (count + 1) / (n_perm + 1)

sig_results = []
y_all = Y_raw.values.ravel()
compare_windows = ["Broad Structure", "Core CH", "Narrow CH"]

for scen in compare_windows:
    for m in m_list:
        preds_s = results[scen][m]['oof_preds']
        preds_f = results["Full Spectrum"][m]['oof_preds']
        mean_diff, ci_l, ci_u, is_boot_sig = bootstrap_r2_diff(y_all, preds_s, preds_f)
        obs_diff, perm_pval = permutation_test_r2(y_all, preds_s, preds_f)


        r2_s_agg = results[scen][m]['oof_r2']
        r2_f_agg = results["Full Spectrum"][m]['oof_r2']

        sig_results.append([m, scen, f"{r2_s_agg:.4f}", f"{r2_f_agg:.4f}",
                            f"[{ci_l:.3f}, {ci_u:.3f}]", "YES" if is_boot_sig else "NO", f"{perm_pval:.4f}", "YES" if perm_pval < 0.05 else "NO"])


PREP_ABBREV = {'none': 'N', 'snv': 'SNV', 'msc': 'MSC', 'deriv1': 'D1', 'deriv2': 'D2', 'wavelet': 'W'}
PARAM_ABBREV = {'n_estimators': 'n', 'max_depth': 'd', 'learning_rate': 'lr', 'subsample': 'subs', 'colsample_bytree': 'cols', 'reg_alpha': 'a', 'reg_lambda': 'l', 'min_child_weight': 'mcw', 'epsilon': 'eps', 'kernel': 'krnl', 'degree': 'deg'}

def abbrev_prep(prep_str):
    for full, short in PREP_ABBREV.items(): prep_str = prep_str.replace(full, short)
    return prep_str

def abbrev_params(p_dict):
    clean = {PARAM_ABBREV.get(k, k): round(v, 4) if isinstance(v, float) else v for k, v in p_dict.items() if not k.startswith('p_') and k != 'use_scaler'}
    return ", ".join([f"{k}:{v}" for k, v in clean.items()])

# --- REPORT GENERATION ---
def draw_styled_table(df, title, footnote=None, fig_size=(11, 8.5), scale_y=2.0):
    fig, ax = plt.subplots(figsize=fig_size); ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    table = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1, scale_y)
    table.auto_set_column_width(col=list(range(len(df.columns))))
    for (row, col), cell in table.get_celld().items():
        if row == 0: cell.set_facecolor('#2B5B84'); cell.set_text_props(weight='bold', color='white')
        else: cell.set_facecolor('#F4F6F9' if row % 2 == 0 else 'white')
    if footnote: ax.text(0.5, 0.02, footnote, transform=ax.transAxes, fontsize=8, ha='center', style='italic')
    return fig

def p_grid(title, yt, res_dict, ml):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    for i, m in enumerate(ml):
        ax = axes.flatten()[i]
        yp = res_dict[m]['oof_preds']
        rmse = res_dict[m]['oof_rmse']
        r2 = res_dict[m]['oof_r2']
        ax.scatter(yt, yp, alpha=0.6, c='blue', edgecolor='k', s=40)
        mx = max(yt.max(), yp.max()); mn = min(yt.min(), yp.min())
        lv = np.linspace(mn, mx, 100)
        ax.plot(lv, lv, 'k--', lw=2, label=f'Aggregate OOF R²: {r2:.3f}')
        ax.fill_between(lv, lv - 1.96 * rmse, lv + 1.96 * rmse, color='gray', alpha=0.2, label='±1.96×RMSE Band')
        ax.set_xlabel('Observed'); ax.set_ylabel('Predicted')
        ax.set_title(f"{m}", fontsize=10)
        ax.legend(loc='upper left', fontsize=8)
    axes.flatten()[-1].axis('off')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

with PdfPages('gasoline_final_zenodo_ready.pdf') as pdf:
    # 1. Spectra Visualization (Full Spectrum Subtle Marker Restored)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(wavs, X_raw.mean(axis=0).values, 'k-', lw=1, label='Mean Spectrum')

    draw_order = ["Broad Structure", "Core CH", "Narrow CH"]
    colors = {'Broad Structure':'#BBDEFB', 'Core CH':'#64B5F6', 'Narrow CH':'#1E88E5'}
    for name in draw_order:
        lo, hi = window_scenarios[name]
        ax.axvspan(lo, hi, alpha=0.6, color=colors[name], label=f'{name} ({sum((wavs >= lo) & (wavs <= hi))} feat.)')

    ax.relim(); ax.autoscale_view()
    y_max = ax.get_ylim()[1]

    # Subtle dashed indicator for Full Spectrum
    ax.axhline(y=y_max*0.98, color='gray', lw=1, ls='--', alpha=0.8)
    ax.text(wavs.mean(), y_max*0.99, f'Full Spectrum (all {len(wavs)} features)', ha='center', va='bottom', fontsize=8, color='gray')
    ax.set_ylim(bottom=ax.get_ylim()[0], top=y_max*1.08) # Make room for text

    ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('Absorbance'); ax.set_title('Spectral Feature Selection Windows')
    ax.legend(fontsize=8, loc='upper left')
    pdf.savefig(fig); plt.close()

    # 2. Parsimony Curve
    fig, ax = plt.subplots(figsize=(10, 6))
    n_feats_list = [sum((wavs >= lo) & (wavs <= hi)) for _, (lo, hi) in window_scenarios.items()]
    for m in m_list:
        r2_vals = [np.mean(results[scen][m]['fold_r2s']) for scen in window_scenarios.keys()]
        r2_stds = [np.std(results[scen][m]['fold_r2s']) for scen in window_scenarios.keys()]
        ax.errorbar(n_feats_list, r2_vals, yerr=r2_stds, fmt='o-', label=m, markersize=8, capsize=4)
    ax.set_xlabel('Number of Spectral Features')
    ax.set_ylabel('Mean Fold R² (Unseen Test)')
    ax.set_title('Parsimony Curve: Model Performance vs Feature Count')
    ax.text(0.02, 0.02, "Error bars show SD of per-fold R² (N=5 folds), establishing stability across data partitions.", transform=ax.transAxes, fontsize=8, style='italic')
    ax.invert_xaxis(); ax.set_ylim(top=1.0); ax.legend()
    pdf.savefig(fig); plt.close()

    # 3. Bar Chart: Computational Cost
    fig, ax = plt.subplots(figsize=(10, 6))
    x_indices = np.arange(len(window_scenarios))
    width = 0.15
    for i, m in enumerate(m_list):
        times = [results[scen][m]['train_time'] for scen in window_scenarios.keys()]
        ax.bar(x_indices + i*width - width*2, times, width, label=m)
    ax.set_xticks(x_indices)
    ax.set_xticklabels(list(window_scenarios.keys()), rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('Total Training Time (s)')
    ax.set_title('Computational Cost: Nested CV Training Time by Model')
    ax.legend()
    pdf.savefig(fig); plt.close()

    # 4. Table 1: All Windows Summary
    cv_data = []
    for scen in window_scenarios.keys():
        for m in m_list:
            d = results[scen][m]
            cv_data.append([
                m, scen,
                f"{np.mean(d['fold_r2s']):.3f} ± {np.std(d['fold_r2s']):.3f}",
                f"{np.mean(d['fold_rmses']):.3f} ± {np.std(d['fold_rmses']):.3f}",
                f"{np.mean(d['fold_maes']):.3f} ± {np.std(d['fold_maes']):.3f}",
                f"{d['train_time']:.1f}s"
            ])
    df_cv = pd.DataFrame(cv_data, columns=['Model', 'Spectral Window', 'Fold R² (Mean ± SD)', 'Fold RMSE (Mean ± SD)', 'Fold MAE (Mean ± SD)', 'Train Time'])
    pdf.savefig(draw_styled_table(df_cv, "Table 1: Unbiased Nested CV Performance Across Window Sizes", footnote="Metrics strictly show the Mean ± Standard Deviation across the 5 blinded outer folds.", fig_size=(12, 10), scale_y=1.8)); plt.close()

    # 5. Table 2: Significance
    df_sig = pd.DataFrame(sig_results, columns=['Model', 'Reduced Window', 'Aggregate R² (Window)', 'Aggregate R² (Full)', 'Bootstrap 95% CI', 'Boot Sig?', 'Perm. p-val', 'Perm Sig?'])
    pdf.savefig(draw_styled_table(df_sig, "Table 2: Statistical Significance (Reduced Windows vs Full Spectrum)", footnote="One-sided permutation test: H0 is that Reduced Window <= Full. Tests compute differences on aggregate pooled OOF predictions.", fig_size=(12, 8.5), scale_y=1.8)); plt.close()

    # 6. Table 3: Preprocessing Stability
    prep_data = []
    for scen in window_scenarios.keys():
        for m in m_list:
            preps = " | ".join([abbrev_prep(p) for p in results[scen][m]['best_preps']])
            l1_note = f"{np.mean(results[scen][m]['l1_ratios']):.4f}" if m == 'ElasticNet' else "-"
            prep_data.append([m, scen, preps, l1_note])
    df_prep = pd.DataFrame(prep_data, columns=['Model', 'Scenario', 'Opt Preprocessing Selected (per Outer Fold)', 'Mean L1'])
    pdf.savefig(draw_styled_table(df_prep, "Table 3: Preprocessing Stability (Nested CV Folds)", footnote="Abbreviations: N=none, SNV=snv, MSC=msc, D1=deriv1, D2=deriv2, W=wavelet. (+S/-S) indicates Standard Scaler.", fig_size=(12, 11), scale_y=2.0)); plt.close()

    # 7. Table 4: Final Production Hyperparameters
    prod_data = []
    for scen in window_scenarios.keys():
        for m in m_list:
            bp = production_params[scen][m]
            p_str = abbrev_prep(bp.get('p_method', 'none')) + f" ({'+S' if bp.get('use_scaler', True) else '-S'})"
            h_str = abbrev_params(bp)
            prod_data.append([m, scen, p_str, h_str])

    df_prod = pd.DataFrame(prod_data, columns=['Model', 'Scenario', 'Final Deployment Preprocessing', 'Final Model Hyperparameters'])
    pdf.savefig(draw_styled_table(df_prod, "Table 4: Final Production Parameters (Optimized on 100% of Data)", footnote="Definitive parameters for deployment. See Table 3 for preprocessing abbreviations.", fig_size=(13, 11), scale_y=2.5)); plt.close()

    # 8. Scatter Plots
    for scen in window_scenarios.keys():
        pdf.savefig(p_grid(f"OOF Diagnostics: {scen}", y_all, results[scen], m_list))
        plt.close()

# --- CODE VERIFICATION ---
print("\n--- CODE VERIFICATION ---")
assert results["Full Spectrum"]["PLS"]["oof_r2"] > 0.5, f"FAIL: PLS Full Spectrum R²={results['Full Spectrum']['PLS']['oof_r2']:.3f} (Expected > 0.5)"
for scen in window_scenarios:
    for m in m_list:
        n_preds = len(results[scen][m]['oof_preds'])
        assert n_preds == len(Y_raw), f"FAIL: {scen}/{m} has {n_preds} preds, expected {len(Y_raw)}"
        assert len(results[scen][m]['fold_r2s']) == 5, f"FAIL: {scen}/{m} has {len(results[scen][m]['fold_r2s'])} folds"
print("✅ All verification checks passed.")
print("✅ Final Masterscript Complete: gasoline_final.pdf")


Evaluating Scenario: Full Spectrum (900.0-1700.0 nm)
  Training PLS...
  Training Ridge...
  Training ElasticNet...
  Training SVR...
  Training XGBoost...
  Running Final Production Tuning for Full Spectrum...
    Production tuning PLS...
    Production tuning Ridge...
    Production tuning ElasticNet...
    Production tuning SVR...
    Production tuning XGBoost...

Evaluating Scenario: Broad Structure (1100-1350 nm)
  Training PLS...
  Training Ridge...
  Training ElasticNet...
  Training SVR...
  Training XGBoost...
  Running Final Production Tuning for Broad Structure...
    Production tuning PLS...
    Production tuning Ridge...
    Production tuning ElasticNet...
    Production tuning SVR...
    Production tuning XGBoost...

Evaluating Scenario: Core CH (1150-1250 nm)
  Training PLS...
  Training Ridge...
  Training ElasticNet...
  Training SVR...
  Training XGBoost...
  Running Final Production Tuning for Core CH...
    Production tuning PLS...
    Production tuning Ridge...
  